# Model Tuning - Selected Dataset

In [8]:

from pathlib import Path
import sys
import os
import json

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import optuna
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline

from app.ml.preprocessing import (
    build_preprocessor,
    prepare_features_and_target,
)

## Load Data

In [9]:
DATA_PATH = Path("../data/processed/bank_fraud_poc_sample.csv")

df = pd.read_csv(DATA_PATH)

X, y = prepare_features_and_target(df)

print("Features:", X.shape)
print("Target:", y.shape)
print(y.value_counts(normalize=True) * 100)

Features: (100000, 20)
Target: (100000,)
is_fraud
0    94.474
1     5.526
Name: proportion, dtype: float64


## Train Test Split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (80000, 20)
Test: (20000, 20)


## Evaluation Helper

In [11]:
def evaluate_model(
    model,
    X_test: pd.DataFrame,
    y_test: pd.Series,
) -> dict[str, float]:
    """Evaluate a fitted binary fraud classifier."""

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions),
        "ROC_AUC": roc_auc_score(y_test, probabilities),
    }

## MLflow Setup

In [ ]:
`os.chdir(PROJECT_ROOT)

MLFLOW_DB = PROJECT_ROOT / "mlflow.db"

mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment(
    "fraud_detection_selected_dataset_tuning"
)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow database:", MLFLOW_DB)

2026/09/21 15:34:28 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/21 15:34:28 INFO mlflow.store.db.utils: Updating database tables
2026/09/21 15:34:31 INFO mlflow.tracking.fluent: Experiment with name 'fraud_detection_selected_dataset_tuning' does not exist. Creating a new experiment.


MLflow tracking URI: sqlite:///mlflow.db
MLflow database: c:\Users\Priya Koma\Desktop\AI_ML_Assessments\ai_fraud_detection_poc\mlflow.db


## Untuned Random Forest

In [13]:
rf_preprocessor = build_preprocessor(
    X_train,
    scale_numeric=False,
)

untuned_rf = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=150,
                class_weight="balanced",
                random_state=42,
                n_jobs=2,
            ),
        ),
    ]
)

untuned_rf.fit(X_train, y_train)

untuned_results = evaluate_model(
    untuned_rf,
    X_test,
    y_test,
)

untuned_results

{'Accuracy': 0.94465,
 'Precision': 0.0,
 'Recall': 0.0,
 'F1': 0.0,
 'ROC_AUC': 0.7151311786138926}

In [14]:
with mlflow.start_run(
    run_name="untuned_random_forest_selected_dataset"
):
    mlflow.log_param("model", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("class_weight", "balanced")

    for metric, value in untuned_results.items():
        mlflow.log_metric(metric.lower(), value)

## Grid Search

In [15]:
grid_preprocessor = build_preprocessor(
    X_train,
    scale_numeric=False,
)

grid_pipeline = Pipeline(
    steps=[
        ("preprocessor", grid_preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                class_weight="balanced",
                random_state=42,
                n_jobs=2,
            ),
        ),
    ]
)

grid_params = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 12],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2],
}

grid_search = GridSearchCV(
    estimator=grid_pipeline,
    param_grid=grid_params,
    scoring="f1",
    cv=3,
    n_jobs=2,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print("Best params:")
print(grid_search.best_params_)

print("\nBest CV F1:")
print(grid_search.best_score_)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Best params:
{'classifier__max_depth': 12, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}

Best CV F1:
0.18164268050683638


In [16]:
grid_results = evaluate_model(
    grid_search.best_estimator_,
    X_test,
    y_test,
)

grid_results

{'Accuracy': 0.733,
 'Precision': 0.11010863561038482,
 'Recall': 0.5411764705882353,
 'F1': 0.18298653610771115,
 'ROC_AUC': 0.7148754189322033}

In [17]:
with mlflow.start_run(
    run_name="grid_search_selected_dataset"
):
    mlflow.log_param("method", "GridSearchCV")

    for key, value in grid_search.best_params_.items():
        mlflow.log_param(key, value)

    for metric, value in grid_results.items():
        mlflow.log_metric(metric.lower(), value)

## Random Search

In [18]:
random_preprocessor = build_preprocessor(
    X_train,
    scale_numeric=False,
)

random_pipeline = Pipeline(
    steps=[
        ("preprocessor", random_preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                class_weight="balanced",
                random_state=42,
                n_jobs=2,
            ),
        ),
    ]
)

random_params = {
    "classifier__n_estimators": [100, 150, 200, 250, 300],
    "classifier__max_depth": [None, 8, 12, 16, 20],
    "classifier__min_samples_split": [2, 4, 6, 8],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2"],
}

random_search = RandomizedSearchCV(
    estimator=random_pipeline,
    param_distributions=random_params,
    n_iter=10,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=2,
    verbose=1,
)

random_search.fit(X_train, y_train)

print("Best params:")
print(random_search.best_params_)

print("\nBest CV F1:")
print(random_search.best_score_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params:
{'classifier__n_estimators': 300, 'classifier__min_samples_split': 4, 'classifier__min_samples_leaf': 4, 'classifier__max_features': 'log2', 'classifier__max_depth': 16}

Best CV F1:
0.1864242265599888


In [19]:
random_results = evaluate_model(
    random_search.best_estimator_,
    X_test,
    y_test,
)

random_results

{'Accuracy': 0.79975,
 'Precision': 0.12396265560165975,
 'Recall': 0.432579185520362,
 'F1': 0.19270308405563394,
 'ROC_AUC': 0.7148517587668936}

In [20]:
with mlflow.start_run(
    run_name="random_search_selected_dataset"
):
    mlflow.log_param("method", "RandomizedSearchCV")

    for key, value in random_search.best_params_.items():
        mlflow.log_param(key, value)

    for metric, value in random_results.items():
        mlflow.log_metric(metric.lower(), value)

## Bayesian Optimization

In [21]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

In [22]:
def objective(trial: optuna.Trial) -> float:
    """Optimize Random Forest hyperparameters using cross-validated F1."""

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            300,
            step=50,
        ),
        "max_depth": trial.suggest_categorical(
            "max_depth",
            [None, 8, 12, 16, 20],
        ),
        "min_samples_split": trial.suggest_int(
            "min_samples_split",
            2,
            8,
        ),
        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf",
            1,
            4,
        ),
        "max_features": trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2"],
        ),
    }

    preprocessor = build_preprocessor(
        X_train,
        scale_numeric=False,
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                RandomForestClassifier(
                    **params,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=2,
                ),
            ),
        ]
    )

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        scoring="f1",
        cv=cv,
        n_jobs=2,
    )

    return scores.mean()

In [23]:
study = optuna.create_study(
    direction="maximize",
)

study.optimize(
    objective,
    n_trials=15,
)

print("Best score:")
print(study.best_value)

print("\nBest parameters:")
print(study.best_params)

[I 2026-09-21 16:00:24,085] A new study created in memory with name: no-name-13a8b25a-4729-4fb2-83da-9faeee2f8daf
[I 2026-09-21 16:00:46,504] Trial 0 finished with value: 0.17055492742641923 and parameters: {'n_estimators': 300, 'max_depth': 8, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.17055492742641923.
[I 2026-09-21 16:01:23,053] Trial 1 finished with value: 0.18247900751394733 and parameters: {'n_estimators': 200, 'max_depth': 12, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.18247900751394733.
[I 2026-09-21 16:02:04,453] Trial 2 finished with value: 0.16650979788252407 and parameters: {'n_estimators': 150, 'max_depth': 16, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.18247900751394733.
[I 2026-09-21 16:02:29,194] Trial 3 finished with value: 0.17464217296257104 and parameters: {'n_estimators': 300, 'max_depth': 

Best score:
0.1836436672894459

Best parameters:
{'n_estimators': 200, 'max_depth': 12, 'min_samples_split': 8, 'min_samples_leaf': 2, 'max_features': 'log2'}


## Evaluate Bayesian Model

In [24]:
bayesian_preprocessor = build_preprocessor(
    X_train,
    scale_numeric=False,
)

bayesian_rf = Pipeline(
    steps=[
        ("preprocessor", bayesian_preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                **study.best_params,
                class_weight="balanced",
                random_state=42,
                n_jobs=2,
            ),
        ),
    ]
)

bayesian_rf.fit(X_train, y_train)

bayesian_results = evaluate_model(
    bayesian_rf,
    X_test,
    y_test,
)

bayesian_results

{'Accuracy': 0.72445,
 'Precision': 0.10856432125088841,
 'Recall': 0.5529411764705883,
 'F1': 0.18149413337293926,
 'ROC_AUC': 0.7156985436306141}

In [25]:
with mlflow.start_run(
    run_name="bayesian_optuna_selected_dataset"
):
    mlflow.log_param("method", "Optuna")

    for key, value in study.best_params.items():
        mlflow.log_param(key, value)

    for metric, value in bayesian_results.items():
        mlflow.log_metric(metric.lower(), value)

## Model Comparison

In [26]:
logistic_results = {
    "Accuracy": 0.7201,
    "Precision": 0.1094,
    "Recall": 0.5692,
    "F1": 0.1835,
    "ROC_AUC": 0.7106,
}

In [27]:
comparison = pd.DataFrame(
    [
        {
            "Method": "Logistic Regression Baseline",
            **logistic_results,
            "Best_Params": "Baseline",
        },
        {
            "Method": "Untuned Random Forest",
            **untuned_results,
            "Best_Params": "Default baseline configuration",
        },
        {
            "Method": "Grid Search Random Forest",
            **grid_results,
            "Best_Params": json.dumps(
                grid_search.best_params_
            ),
        },
        {
            "Method": "Random Search Random Forest",
            **random_results,
            "Best_Params": json.dumps(
                random_search.best_params_
            ),
        },
        {
            "Method": "Bayesian Optuna Random Forest",
            **bayesian_results,
            "Best_Params": json.dumps(
                study.best_params
            ),
        },
    ]
)

comparison.sort_values(
    "F1",
    ascending=False,
)

,Method,Accuracy,Precision,Recall,F1,ROC_AUC,Best_Params
3,Random Search Random Forest,0.79975,0.123963,0.432579,0.192703,0.714852,"{""classifier__n_estimators"": 300, ""classifier_..."
0,Logistic Regression Baseline,0.72010,0.109400,0.569200,0.183500,0.710600,Baseline
2,Grid Search Random Forest,0.73300,0.110109,0.541176,0.182987,0.714875,"{""classifier__max_depth"": 12, ""classifier__min..."
4,Bayesian Optuna Random Forest,0.72445,0.108564,0.552941,0.181494,0.715699,"{""n_estimators"": 200, ""max_depth"": 12, ""min_sa..."
1,Untuned Random Forest,0.94465,0.000000,0.000000,0.000000,0.715131,Default baseline configuration


## Tuning Conclusions